# Advancing Voice Intelligence — Cookbook

This notebook contains the cookbook converted from markdown. It includes examples, patterns, and best practices for using the new OpenAI voice models described in the announcement at https://openai.com/index/advancing-voice-intelligence-with-new-models-in-the-api/. Replace the example model IDs with the exact model names from the API docs or announcement page.

---

## Quick Start

- **Prereqs:** Python 3.8+, Node 18+, FFmpeg for local audio conversion.
- **Install (Python):**

```bash
pip install openai soundfile requests
# or the official SDK/package the announcement references
```

- **Install (Node):**

```bash
npm install openai node-fetch
```

## Authentication

Set your API key in the environment:

```bash
export OPENAI_API_KEY="sk-..."
# Windows PowerShell:
$env:OPENAI_API_KEY = "sk-..."
```

---




### Speech-to-Text (Batch upload) — Python example

In [ ]:
# Install required packages if not already installed:
# !pip install openai soundfile requests

from openai import OpenAI

# Initialize client (ensure OPENAI_API_KEY is set in your environment)
client = OpenAI()

# Replace 'voice_sample.wav' with your local file path and model id with the announced one
with open('voice_sample.wav', 'rb') as f:
    transcript = client.audio.transcriptions.create(
        file=f,
        model='voice-transcribe-1',  # replace with actual model id
        language='en',
        temperature=0.0
    )

print('Transcript:')
try:
    print(transcript.text)
except Exception as e:
    print('Response object:', transcript)
    print('Note: The SDK surface may differ; adapt to the official client API.')

### Text-to-Speech (TTS) — Python example

In [ ]:
from openai import OpenAI

client = OpenAI()

# Replace model and voice names with actual values from the announcement/docs
resp = client.audio.speech.create(
    model='voice-tts-1',
    voice='alloy',
    input='Hello — this is a quick TTS example. Speak clearly and warmly.',
    format='wav'
)

# Save result to file (SDK may return bytes or a streaming object; adapt if necessary)
try:
    with open('tts_output.wav', 'wb') as f:
        f.write(resp.audio)
    print('Wrote tts_output.wav')
except Exception as e:
    print('Could not write audio directly; inspect resp object:', type(resp))
    print(resp)

### FFmpeg conversion tip (run in a shell)

In [ ]:
%%bash
ffmpeg -i input.mp3 -ar 16000 -ac 1 -c:a pcm_s16le out.wav || echo 'ffmpeg not available in this environment'

### Node.js examples and instructions

Run Node examples outside the notebook. Create a file like `tts.js` or `stt.js` and run with `node tts.js`. Example JavaScript snippets are below:

```js
import fs from 'fs';
import OpenAI from 'openai';

const client = new OpenAI({ apiKey: process.env.OPENAI_API_KEY });

// Transcription example
const transcription = await client.audio.transcriptions.create({
  file: fs.createReadStream('voice_sample.wav'),
  model: 'voice-transcribe-1'
});
console.log(transcription.text);

// TTS example
const speech = await client.audio.speech.create({
  model: 'voice-tts-1',
  voice: 'alloy',
  input: 'Short announcement: lunch in 10 minutes.',
  format: 'mp3'
});

fs.writeFileSync('announcement.mp3', Buffer.from(await speech.arrayBuffer()));
```

---

## 3) Low-latency & Real-time patterns

There are two typical patterns:
- Server-side streaming: send chunks of recognized audio and receive partial transcripts or partial TTS audio for immediate playback.
- Peer-to-peer pipe: capture microphone frames, encode (Opus/PCM), and send over a WebSocket to the model endpoint.

Design notes:
- Use small frames (20–60ms) for low latency.
- Use compressed streaming (Opus) for bandwidth-sensitive scenarios.
- Implement jitter buffers and packet re-ordering on playback.

Simplified pseudo-workflow:
1. Capture microphone frames, encode to Opus.
2. Send frames over WebSocket to model endpoint.
3. Receive partial recognition / partial TTS packets.
4. Mix and play audio while continuing to stream.

## 4) Voice Conversion, Cloning, and Persona

- If the API supports voice cloning, follow the model's consent and dataset requirements.
- Always obtain explicit consent from the voice owner and follow any legal/regulatory requirements.

Recipe: Creating a persona-aware TTS
1. Choose or fine-tune (if available) a base voice.
2. Provide a persona prompt: short bio, speaking style, and example sample utterances.
3. Use small refinement prompts each session to adapt to context (e.g., "Speak like a calm teacher explaining to a child").

## 5) Audio formats & encoding

- Preferred upload formats for best fidelity: WAV (PCM 16-bit), FLAC.
- Compressed for streaming: Opus in Ogg or raw Opus.
- Sample rates: 16kHz for voice telephony; 48kHz for high-quality audio.

Conversion tip (FFmpeg):
```bash
ffmpeg -i input.mp3 -ar 16000 -ac 1 -c:a pcm_s16le out.wav
```

## 6) Prompt & Interaction Design for Voice

- Keep spoken prompts short and scannable.
- Use confirmation strategies: repeat crucial facts back to the user.
- For multi-turn voice agents, include a short system instruction describing expected behavior and allowed latency.
- Use turn markers (beep or short phrase) to indicate when the system is listening vs. speaking.

## 7) Testing & Evaluation

- Automated metrics: WER (word error rate) for STT, MOS (mean opinion score) proxies for TTS.
- Human tests: A/B listening tests, naturalness, intelligibility, latency perceived by users.
- Logging: store anonymized transcripts, latency metrics, and audio quality indicators.

## 8) Safety, Privacy & Compliance

- Avoid storing raw audio unless necessary; store derived transcripts with care.
- Redact PII where possible before persisting.
- Ensure voice-cloning workflows have explicit consent and keep audit logs.

## 9) Troubleshooting Common Issues

- Issue: Muffled transcription — ensure recording gain is sufficient and sample rate meets model expectations.
- Issue: Robotic TTS — try a different voice setting or add prosody hints / SSML.
- Issue: High latency — reduce frame size, use local prefetching, or switch to lower-bandwidth codec.

## 10) Resources & Links

- Official announcement: https://openai.com/index/advancing-voice-intelligence-with-new-models-in-the-api/
- OpenAI audio & voice API docs: (link to official docs — replace with the exact doc URL from the announcement)

---

### Extras: Evaluation snippets and quick utilities

Below are small helper snippets you can run in the notebook to compute simple evaluation metrics and inspect audio properties.

```python
# Example: compute simple word error rate (WER) between reference and hypothesis
def wer(ref, hyp):
    r = ref.split()
    h = hyp.split()
    # simple dynamic programming edit distance
    d = [[0]*(len(h)+1) for _ in range(len(r)+1)]
    for i in range(len(r)+1): d[i][0] = i
    for j in range(len(h)+1): d[0][j] = j
    for i in range(1, len(r)+1):
        for j in range(1, len(h)+1):
            cost = 0 if r[i-1]==h[j-1] else 1
            d[i][j] = min(d[i-1][j]+1, d[i][j-1]+1, d[i-1][j-1]+cost)
    return d[len(r)][len(h)] / max(1, len(r))
```

```python
# Example usage: wer('this is a test', 'this is test') -> 0.25
print('WER example:', wer('this is a test', 'this is test'))
